# The reserved period: checks, then one scoring

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/reserved_evaluation.ipynb)

Two frozen protocols share the period 2024-01-03 to 2026-10-12: `protocols/volatility_v2.yaml` and `protocols/portfolio_v2.yaml`. **It is reserved for these specifications but previously exposed through related research**; `protocols/reserved_period_disclosure.md` lists every prior use.

This notebook runs in steps. Set `STEP` below, then Runtime → Run all. Each step includes the ones before it, and finished files are kept. Results print here; nothing is downloaded. Paste the output into the chat.

| STEP | What it does | Needs | Time |
|---|---|---|---:|
| **1** | The data checks: SPY and BIL against the recorded returns, official-open coverage, whether missing opens cluster on extreme days, quoted spreads. **Scores nothing.** For the reserved period it prints coverage counts only. | Colab Secrets | about 20 minutes |
| **2** | Builds sessions up to the lock and runs the baseline gate on the official open. **No model uses PE, and nothing after 2024-01-02 is read.** | Google Drive (the 5-minute cache) | about 20 minutes more |
| **3** | **Locked. Scores the reserved period, once:** volatility first, then the portfolio. It cannot be undone, and is not to run until Kshitiz decides. | `CONFIRM` below, typed exactly | about 45 minutes more |

**Don't save this notebook to GitHub from an old copy.** The code it runs is hash-checked by `tests/test_frozen.py`, so a change to a protocol or its code shows up.

In [8]:
STEP = 1       # 1 or 2. STEP 3 is locked until Kshitiz decides
CONFIRM = ''   # for STEP 3 only: 'score the reserved period'

In [9]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/entropy-gate-pipeline' if IN_COLAB else os.environ.get('PORTFOLIO_REPO', os.getcwd())
if IN_COLAB:
    if os.path.isdir(f'{REPO}/.git'):
        !cd {REPO} && git pull -q --ff-only
    else:
        !git clone -q https://github.com/Gaire-commits/entropy-gate-pipeline.git {REPO}
    %cd {REPO}
    !git log --oneline -1
    from google.colab import userdata
    try:
        with open('.env', 'w') as f:
            f.write(f"APCA_API_KEY_ID={userdata.get('APCA_API_KEY_ID')}\nAPCA_API_SECRET_KEY={userdata.get('APCA_API_SECRET_KEY')}\n")
        print('keys written to a local .env (never printed, never committed)')
    except Exception as exc:
        print('No Alpaca keys in Colab Secrets:', type(exc).__name__, '- add them from the key icon, then run this cell again.')
else:
    os.chdir(REPO)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
sys.path[:0] = [REPO, f'{REPO}/scripts']

from pathlib import Path
REF = os.environ.get('RESERVED_REF', '/content/reserved/reference')
WORK = os.environ.get('RESERVED_WORK', '/content/reserved')
DRIVE = '/content/drive/MyDrive/entropy-gate-pipeline-data/data'
Path(REF).mkdir(parents=True, exist_ok=True)

/content/entropy-gate-pipeline
b468972 (HEAD -> main, origin/main, origin/HEAD) Created using Colab
keys written to a local .env (never printed, never committed)


## Step 1: the data checks

In [10]:
!python scripts/fetch_reference.py --universe universe/sp500.csv --start 2016-01-04 --end 2026-10-13 --out {REF} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
!python scripts/official_prices.py --reference {REF}
!python scripts/protocol_checks.py --reference {REF}

kept /content/reserved/reference/daily/daily_sip_split_nomap.parquet
kept /content/reserved/reference/corporate_actions.parquet
kept /content/reserved/reference/auctions.parquet
done in 0s, 0 requests; manifest at /content/reserved/reference/manifest.json
6,497,884 auction prints for 502 symbols; 1,307,857 vendor bars with a trade, 100.00% with a listing exchange

listing exchanges (symbol-months; Nasdaq under Q): {'N': 44159, 'Q': 18534, 'Z': 67, 'V': 6, 'P': 1, nan: 1}

coverage by year (shares of vendor bars; counts only, so every year is shown):
        bars  with_listing  with_official_open  with_official_close
date                                                               
2016  115792           1.0              0.9991               0.9998
2017  116340           1.0              0.9992               0.9998
2018  117523           1.0              0.9979               0.9995
2019  119593           1.0              0.9975               0.9959
2020  121494           1.0          

## Step 2: the baseline gate on the official open

In [11]:
if STEP >= 2:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    if not Path(f'{WORK}/sessions_to_lock.parquet').exists():
        !python scripts/build_sessions.py --sip {DRIVE}/bars_sip --until 2024-01-02 --out {WORK}/sessions_to_lock.parquet
    !python scripts/vol_forecast.py --stage baseline --sessions {WORK}/sessions_to_lock.parquet --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- skipped')

STEP is 1 - skipped


## Step 3: locked

This cell scores the reserved period, once. It does nothing unless `STEP = 3` and `CONFIRM` is typed exactly, and it is not to run until Kshitiz decides. Record the outcome in `RESULTS.md` whatever it is.

In [12]:
if STEP >= 3:
    assert CONFIRM == 'score the reserved period', "set CONFIRM exactly to run this step"
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    if not Path(f'{WORK}/sessions_full.parquet').exists():
        !python scripts/build_sessions.py --sip {DRIVE}/bars_sip --until 2026-10-12 --out {WORK}/sessions_full.parquet
    if not Path(f'{REF}/rv5.parquet').exists():
        !python scripts/vol_forecast.py --stage rv --sip {DRIVE}/bars_sip --rv {REF}/rv5.parquet
    print('=' * 30, 'VOLATILITY')
    !python scripts/vol_forecast.py --stage reserved --score-reserved --sessions {WORK}/sessions_full.parquet --reference {REF} --official {REF}/official_prices.parquet --rv {REF}/rv5.parquet 2>&1 | grep -v Warning
    print('=' * 30, 'PORTFOLIO')
    !python scripts/reserved_portfolio.py --reference {REF} --score-reserved 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- the reserved period was not scored')

STEP is 1 - the reserved period was not scored


In [13]:
# Why is 2021 short of official opens? Read-only: writes nothing, scores nothing, uses only dates before the lock.
import numpy as np, pandas as pd
REF = '/content/reserved/reference'
LOCK = pd.Timestamp('2024-01-03')
op = pd.read_parquet(f'{REF}/official_prices.parquet', columns=['symbol', 'date', 'listing', 'official_open', 'vendor_close'])
op['date'] = pd.to_datetime(op['date']).dt.normalize()
op = op[op['date'] < LOCK].copy()
a = pd.read_parquet(f'{REF}/auctions.parquet', columns=['symbol', 'date', 'side', 'condition', 'price', 'exchange'])
a['date'] = pd.to_datetime(a['date']).dt.tz_localize(None).dt.normalize()
a = a[a['date'] < LOCK].copy()
op['month'] = op['date'].dt.strftime('%Y-%m')
a['month'] = a['date'].dt.strftime('%Y-%m')
op['missing'] = op['official_open'].isna()
y21 = op[op['date'].dt.year == 2021]
print('A. 2021, share of bars with no official open, by month and by the exchange chosen for the year:')
print(y21.pivot_table(index='month', columns='listing', values='missing', aggfunc='mean').round(3).to_string())
by_day = y21.groupby('date')['missing'].mean()
print(f'\nB. 2021 sessions with over half of stocks missing: {int((by_day > .5).sum())} of {len(by_day)}; largest share on one day {by_day.max():.3f}')
c = a[(a['side'] == 'close') & a['condition'].isin(['M', '6'])].merge(op[['symbol', 'date', 'vendor_close']], on=['symbol', 'date'])
c = c[(1e4 * np.log(c['vendor_close'] / c['price'])).abs() <= 0.5]
print('\nC. Closing prints equal to the vendor close, by exchange and month (2020-07 to 2021-12):')
print(c[c['month'].between('2020-07', '2021-12')].pivot_table(index='month', columns='exchange', values='price', aggfunc='size').fillna(0).astype(int).to_string())
best = c.groupby(['symbol', 'month', 'exchange']).size().rename('n').reset_index().sort_values(['symbol', 'month', 'n'], ascending=[True, True, False]).drop_duplicates(['symbol', 'month'])
opens = a[(a['side'] == 'open') & a['condition'].isin(['Q', 'O'])]
per_month = opens.merge(best[['symbol', 'month', 'exchange']], on=['symbol', 'month', 'exchange'])[['symbol', 'date']].drop_duplicates().assign(per_month=True)
anywhere = opens[['symbol', 'date']].drop_duplicates().assign(any_exchange=True)
x = op.merge(per_month, on=['symbol', 'date'], how='left').merge(anywhere, on=['symbol', 'date'], how='left')
g = x.groupby(x['date'].dt.year)
print('\nD. Coverage of all vendor bars by year: as run (exchange chosen per year); if chosen per month; with a print on any exchange:')
print(pd.DataFrame({'as_run': 1 - g['missing'].mean(), 'per_month': g['per_month'].count() / g.size(), 'any_exchange': g['any_exchange'].count() / g.size()}).round(4).to_string())
names = ['EXPD', 'VRT', 'APA', 'IBKR', 'TPL', 'FISV', 'FITB', 'HON']
b = best[best['symbol'].isin(names) & best['month'].between('2021-01', '2021-12')]
print('\nE. Exchange matching the vendor close most often, per month of 2021, for the stocks missing most:')
print(b.assign(month=b['month'].str[5:]).pivot(index='symbol', columns='month', values='exchange').fillna('-').to_string())
print('\nchosen per year, 2021:', op[op['symbol'].isin(names) & (op['date'].dt.year == 2021)].groupby('symbol')['listing'].first().to_dict())

A. 2021, share of bars with no official open, by month and by the exchange chosen for the year:
listing      N      Q    Z
month                     
2021-01  0.000  0.004  NaN
2021-02  0.000  0.005  NaN
2021-03  0.000  0.005  NaN
2021-04  0.007  0.000  0.0
2021-05  0.001  0.000  0.0
2021-06  0.000  0.000  0.0
2021-07  0.000  0.000  0.0
2021-08  0.000  0.000  0.0
2021-09  0.000  0.000  0.0
2021-10  0.001  0.000  0.0
2021-11  0.000  0.000  0.0
2021-12  0.001  0.000  0.0

B. 2021 sessions with over half of stocks missing: 0 of 252; largest share on one day 0.037

C. Closing prints equal to the vendor close, by exchange and month (2020-07 to 2021-12):
exchange   A      N     P     Q     T   Z
month                                    
2020-07    0   7523     3     0  3033   5
2020-08    0   7181     3     0  2890   4
2020-09    0   7172     6     0  2860   4
2020-10    0   7488     2     0  3040   2
2020-11    0   7050    51     0  2784   3
2020-12    0   7491     6     0  3091   1
2021-01

In [14]:
!git pull -q --ff-only && git log --oneline -1
!python scripts/official_listing_candidates.py --reference /content/reserved/reference 2>&1 | grep -v Warning

b468972 (HEAD -> main, origin/main, origin/HEAD) Created using Colab
A. Official-open coverage of usable bars, by year (counts only):
      as run  per year, T and Q as one  per month  per month, T and Q as one
date                                                                        
2016  1.0000                    1.0000     1.0000                     1.0000
2017  0.9998                    0.9998     0.9998                     0.9998
2018  0.9989                    0.9978     0.9989                     0.9989
2019  0.9998                    0.9992     0.9998                     0.9998
2020  0.9992                    0.9973     0.9992                     0.9992
2021  0.9990                    0.9988     0.9990                     0.9990
2022  0.9994                    0.9994     0.9991                     0.9994
2023  0.9978                    0.9959     0.9900                     0.9978
2024  1.0000                    0.9998     1.0000                     1.0000
2025  1.0000       